In [ ]:
# Import tkinter for GUI and ttk for improved widgets
import tkinter as tk
from tkinter import ttk, messagebox

# Define the RestaurantOrderManagementApp class
class RestaurantOrderManagement:
    # Initialize the application
    def __init__(self, root):
        self.root = root  # Store the main window instance
        self.root.title("Restaurant Management App")  # Set the title of the GUI window

        # A dictionary to store the menu items and their prices (in USD)
        self.menu_items = {
            "FRIES MEAL": 2,
            "LUNCH MEAL": 2,
            "BURGER MEAL": 3,
            "PIZZA MEAL": 4,
            "CHEESE BURGER": 2.5,
            "DRINKS": 1
        }

        self.exchange_rate = 82  # Define the exchange rate for currency conversion (e.g., USD to INR)

        self.setup_background(root)  # Call a method to set up the background image for the window

        # Create a frame (container) to hold all the main widgets
        frame = ttk.Frame(root)
        # Place the frame in the center of the root window
        frame.place(relx=0.5, rely=0.5, anchor=tk.CENTER)

        # Create a heading label for the application
        ttk.Label(
            frame,
            text="Restaurant Order Management",
            font=("Arial", 20, "bold")
        ).grid(row=0, columnspan=3, padx=10, pady=10) # Place heading in the grid layout

        self.menu_labels = {}       # Dictionary to store references to menu item labels (for price updates)
        self.menu_quantities = {}   # Dictionary to store references to quantity entry widgets

        # Loop through each menu item to create a label and an entry field for quantity
        for i, (item, price) in enumerate(self.menu_items.items(), start=1):
            # Create a label for each menu item, showing its name and default price
            label = ttk.Label(
                frame,
                text=f"{item} (${price}):", # f-string for formatted text
                font=("Arial", 12)
            )
            label.grid(row=i, column=0, padx=10, pady=5) # Place the label in the grid
            self.menu_labels[item] = label # Store reference to the label

            # Create an entry widget for the user to input quantity
            quantity_entry = ttk.Entry(frame, width=5)
            quantity_entry.grid(row=i, column=1, padx=10, pady=5) # Place the entry widget
            self.menu_quantities[item] = quantity_entry # Store reference to the entry widget

        # Currency selection section
        self.currency_var = tk.StringVar() # Tkinter variable to hold the selected currency
        # Create a label for currency selection
        ttk.Label(
            frame,
            text="Currency:",
            font=("Arial", 12)
        ).grid(
            row=len(self.menu_items) + 1, # Place below all menu items
            column=0,
            padx=10,
            pady=5
        )

        # Create a dropdown (Combobox) for currency selection
        currency_dropdown = ttk.Combobox(
            frame,
            textvariable=self.currency_var, # Link to the StringVar
            state="readonly", # Make it non-editable
            width=18,
            values=("USD", "INR") # Available currency options
        )
        currency_dropdown.grid(
            row=len(self.menu_items) + 1,
            column=1,
            padx=10,
            pady=5
        )
        currency_dropdown.current(0)  # Set the default selected currency to the first item (USD)
        # Trace changes in currency_var to update menu prices automatically
        self.currency_var.trace("w", self.update_menu_prices)

        # Button to place the order
        order_button = ttk.Button(
            frame,
            text="Place Order",
            command=self.place_order # Link button click to place_order method
        )
        order_button.grid(
            row=len(self.menu_items) + 2, # Place below currency selection
            columnspan=3, # Span across columns
            padx=10,
            pady=10
        )

    # Method to set up the background image for the application window
    def setup_background(self, root):
        bg_width, bg_height = 800, 600 # Define desired background dimensions
        canvas = tk.Canvas(root, width=bg_width, height=bg_height) # Create a Canvas widget
        canvas.pack() # Pack the canvas to make it visible

        # Load the background image from file
        # Note: PhotoImage primarily supports GIF and PGM/PPM. For other formats (PNG, JPG), Pillow (PIL) is needed.
        original_image = tk.PhotoImage(file="background.png")
        # Resize the image using subsampling to fit the background dimensions
        background_image = original_image.subsample(
            original_image.width() // bg_width,
            original_image.height() // bg_height
        )

        # Place the resized background image on the canvas
        canvas.create_image(0, 0, anchor=tk.NW, image=background_image)
        # Keep a reference to the image to prevent it from being garbage-collected
        canvas.image = background_image

    # Method to update the menu prices displayed on the GUI based on the selected currency
    def update_menu_prices(self, *args): # *args is used to accept extra arguments passed by .trace()
        currency = self.currency_var.get() # Get the currently selected currency
        symbol = "₹" if currency == "INR" else "$" # Determine the currency symbol
        rate = self.exchange_rate if currency == "INR" else 1 # Determine the exchange rate

        # Loop through each menu item label and update its text with the new price
        for item, label in self.menu_labels.items():
            price = self.menu_items[item] * rate # Calculate the price in the selected currency
            label.config(text=f"{item} ({symbol}{price}):") # Update the label's text

    # Method to process the order when the "Place Order" button is clicked
    def place_order(self):
        total_cost = 0 # Initialize total cost of the order
        order_summary = "Order Summary:\n" # Start building the order summary string
        currency = self.currency_var.get() # Get the selected currency
        symbol = "₹" if currency == "INR" else "$" # Get the currency symbol
        rate = self.exchange_rate if currency == "INR" else 1 # Get the exchange rate

        # Iterate through each menu item's quantity entry field
        for item, entry in self.menu_quantities.items():
            quantity = entry.get() # Get the text from the quantity entry
            if quantity.isdigit(): # Check if the input is a valid number
                quantity = int(quantity) # Convert quantity to an integer
                price = self.menu_items[item] * rate # Calculate item price with exchange rate
                cost = quantity * price # Calculate cost for this item
                total_cost += cost # Add to the overall total cost

                if quantity > 0: # If quantity is greater than 0, add to summary
                    order_summary += (
                        f"{item}: {quantity} x {symbol}{price} = {symbol}{cost}\n"
                    )

        # Display the order summary or an error message
        if total_cost > 0: # If at least one item was ordered
            order_summary += f"\nTotal Cost: {symbol}{total_cost}"
            # Show an information message box with the order details
            messagebox.showinfo("Order Placed", order_summary)
        else: # If no items were ordered
            # Show an error message box
            messagebox.showerror("Error", "Please order at least one item.")

# Main block to run the application when the script is executed
if __name__ == "__main__":
    root = tk.Tk() # Create the main Tkinter window
    app = RestaurantOrderManagement(root) # Create an instance of the RestaurantManagementApp class
    root.geometry("800x600")  # Set the initial size of the window
    root.mainloop()           # Start the Tkinter event loop, keeping the window open and responsive